# Plain source configuration and WR export

Edit `trainer/config.json` as plain JSON. Use this notebook to inspect or update that source, then explicitly export an encoded copy for the standalone WR trainer. The source stays plain and unchanged during export.

Run the setup and helper-definition cells first. They do not read config values, write files, or request a decode key. All execution examples are commented, so running every cell is harmless until you call a helper yourself.

Select the **LRAEP trainer** kernel or this project's `.venv/bin/python` interpreter. Install dependencies from the project folder if needed:

```bash
uv pip install --python .venv/bin/python -r requirements.txt
```

`show_config()` explicitly prints credentials into notebook output. Clear those outputs before saving or sharing. Setup disables IPython history logging; notebook source and output saving still follow your editor's settings.


In [ ]:
import json
import sys
from getpass import getpass
from pathlib import Path

try:
    shell = get_ipython()
except NameError:
    shell = None
if shell is not None:
    shell.history_manager.enabled = False
    shell.history_manager.db_log_output = False


def find_project_root():
    here = Path.cwd().resolve()
    for directory in (here, *here.parents):
        for candidate in (directory, directory / "trainer"):
            if (candidate / "ann_trainer.py").is_file() and (
                candidate / "src" / "LRAEP_trainer" / "config.py"
            ).is_file():
                return candidate
    raise RuntimeError("Open this notebook from the trainer project or its parent folder.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from LRAEP_trainer.config import read_config, save_config, update_config
from LRAEP_trainer.config_transfer import export_wr_config as write_wr_config

CONFIG_PATH = PROJECT_ROOT / "config.json"
WR_CONFIG_PATH = PROJECT_ROOT.parent / "LRAEP-WR-Trainer" / "config.json"


In [ ]:
def show_config():
    """Print the plain source only when explicitly called."""
    print(json.dumps(read_config(CONFIG_PATH), indent=2, ensure_ascii=False))


def update_from_dict(plain_dict):
    """Deep-merge settings and save the editable source as plain JSON."""
    update_config(CONFIG_PATH, plain_dict)
    print(f"Plain configuration updated: {CONFIG_PATH}")


def replace_from_dict(plain_dict):
    """Validate and replace the complete plain source configuration."""
    save_config(CONFIG_PATH, plain_dict)
    print(f"Plain configuration replaced: {CONFIG_PATH}")


def export_wr_config(output_path=WR_CONFIG_PATH, decode_key=None, save_key=True):
    """Export an encoded WR copy using the exact decode text supplied here."""
    if decode_key is None:
        decode_key = getpass("Decode key text: ")
    output_path = write_wr_config(CONFIG_PATH, output_path, open_text=decode_key,
                                 save_key=save_key)
    print(f"Encoded WR configuration: {output_path}")
    if save_key:
        print(f"Decode key saved in: {output_path.parent / '.env'}")
    return output_path


## Inspect or edit the plain source

Uncomment only the commands you want to run. `update_from_dict` merges nested dictionaries and preserves unspecified fields; `replace_from_dict` accepts a complete configuration including `"version": 1`. Both validate before saving plain JSON. Relative paths remain relative.

Job entries are JSON triples `["main" or "decoy", seconds, maximum_gpu_percentage]` in `training.job_sequence`. Entries run in order and repeat. Positive percentages mine, `0` pauses for that duration, and `-1` exits when the entry begins. See the trainer README for all configuration instructions.


In [ ]:
# show_config()

# update_from_dict({
#     "training": {
#         "job_sequence": [["main", 60, 80], ["decoy", 30, 20]],
#     },
# })

# update_from_dict({"pool": {"password": getpass("New pool password: ")}})


## Export the standalone WR configuration

Fill in `pool.WR`, `pool.WR_wallet`, and enabled VPN settings (`vpn.enabled: true` and `vpn.link`) in the plain source first; export validates them.

Call `export_wr_config()` explicitly to enter arbitrary decode text through a hidden prompt. The text stays local to that call and is not printed or assigned to a notebook global.

The default destination is `LRAEP-WR-Trainer/config.json` beside this project. Export selects pool mode and the WR profile while preserving the source settings and relative paths. It leaves `trainer/config.json` plain and unchanged.

By default, export also writes the exact same UTF-8 text to `open_text` in the destination's adjacent `.env`, preserving unrelated entries. WR uses that entry to decode the exported document. A manual entry has the form `open_text="your chosen decode text"`; the notebook handles quoting and escaping for you.

To choose another destination, supply `output_path`. With `save_key=False`, `.env` remains unchanged and must already contain the matching `open_text` before WR starts. Export prints only output paths. Re-export after making source changes that WR should use.


In [ ]:
# export_wr_config()

# Export to another config path, with an adjacent .env:
# export_wr_config(output_path=PROJECT_ROOT / "exports" / "config.json")
